# INC-014 | Registrar peças muda o tempo médio de atendimento

## Relato do cliente
"O painel mostra, por setor, o total de unidades de peças consumidas em ordens encerradas e o tempo médio entre abertura e encerramento. Depois de lançar peças em uma ordem já encerrada, a média do setor mudou, embora as ordens e suas datas continuassem iguais. Cada ordem encerrada deve ter o mesmo peso na média, inclusive quando não consumiu peças."

## Hipótese inicial
A consulta SQL do painel une as tabelas `ordens_servico` e `itens_ordem` através de um `LEFT JOIN` e calcula a média `AVG(o.encerrada_em - o.aberta_em)` diretamente sobre o conjunto agregado agrupado por setor. Como a cardinalidade entre ordens e itens é de 1 para N ($1:N$), ordens com múltiplos itens de peças são replicadas na junção tantas vezes quantos forem os itens. Dessa forma, ordens com muitas peças possuem peso desproporcional no cálculo da média de tempo (`AVG`). Quando novas peças são adicionadas a uma OS encerrada, a quantidade de linhas duplicadas daquela ordem aumenta, distorcendo indevidamente a média do setor.

In [ ]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd

raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

config = dotenv_values(raiz / "docker" / ".env")

def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=55432,
        dbname="pbl_industria_testes",
        user="aluno",
        password=config.get("POSTGRES_PASSWORD", "pipe101198")
    )

## 1. Análise da consulta SQL fornecida pelo cliente

A consulta original utilizada pelo painel gerencial do cliente:

```sql
SELECT
    e.setor,
    COALESCE(SUM(i.quantidade), 0) AS total_pecas_consumidas,
    AVG(EXTRACT(EPOCH FROM (o.encerrada_em - o.aberta_em)) / 3600.0) AS tempo_medio_horas
FROM industria.ordens_servico o
JOIN industria.equipamentos e ON e.id = o.equipamento_id
LEFT JOIN industria.itens_ordem i ON i.ordem_servico_id = o.id
WHERE o.status = 'ENCERRADA'
GROUP BY e.setor
ORDER BY e.setor;
```

### Por que a média de tempo é distorcida?
- Seja uma OS que demorou **10 horas** para ser concluída e utilizou **4 peças distintas**. O `LEFT JOIN` gera 4 linhas com a mesma duração de 10 horas.
- Seja outra OS do mesmo setor que demorou **2 horas** e não utilizou nenhuma peça. O `LEFT JOIN` gera 1 linha com duração de 2 horas.
- **Média correta das ordens**: $rac{10 + 2}{2} = \mathbf{6.0	ext{ horas}}$.
- **Média calculada pela consulta do cliente**: $rac{10 + 10 + 10 + 10 + 2}{5} = rac{42}{5} = \mathbf{8.4	ext{ horas}}$!
- Ao registrar uma nova peça na primeira OS, a duração de 10 horas é contada mais uma vez na média, elevando o resultado para $rac{52}{6} = 8.67	ext{ horas}$, sem que nenhuma data tenha sofrido alteração!

In [ ]:
# Consulta do painel pelo formato original do cliente
sql_painel_cliente = """
SELECT
    e.setor,
    COALESCE(SUM(i.quantidade), 0) AS total_pecas,
    ROUND(AVG(EXTRACT(EPOCH FROM (o.encerrada_em - o.aberta_em)) / 3600.0)::numeric, 4) AS media_horas_cliente
FROM industria.ordens_servico o
JOIN industria.equipamentos e ON e.id = o.equipamento_id
LEFT JOIN industria.itens_ordem i ON i.ordem_servico_id = o.id
WHERE o.status = 'ENCERRADA'
GROUP BY e.setor
ORDER BY e.setor;
"""

with conectar_testes() as conn:
    df_painel_antes = pd.read_sql(sql_painel_cliente, conn)

print("Painel original do cliente (Antes da inserção de peças):")
display(df_painel_antes)

In [ ]:
# Reprodução experimental da anomalia de média
# 1. Escolher uma OS encerrada do setor 'Usinagem'
with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT o.id, o.numero, e.setor
            FROM industria.ordens_servico o
            JOIN industria.equipamentos e ON e.id = o.equipamento_id
            WHERE o.status = 'ENCERRADA'
            LIMIT 1;
        """)
        os_id, os_num, setor_alvo = cur.fetchone()

print(f"OS selecionada para teste: {os_num} (ID: {os_id}) do setor: {setor_alvo}")

# 2. Inserir peças adicionais na OS já encerrada
with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO industria.itens_ordem
                (ordem_servico_id, componente_id, quantidade, valor_unitario_aplicado)
            VALUES (%s, 1, 5, 20.00), (%s, 2, 10, 30.00);
        """, (os_id, os_id))
        conn.commit()

# 3. Recalcular o painel com a consulta do cliente
with conectar_testes() as conn:
    df_painel_depois = pd.read_sql(sql_painel_cliente, conn)

print("Painel do cliente (Após inserir peças na OS encerrada):")
display(df_painel_depois)

media_antes = df_painel_antes[df_painel_antes["setor"] == setor_alvo]["media_horas_cliente"].iloc[0]
media_depois = df_painel_depois[df_painel_depois["setor"] == setor_alvo]["media_horas_cliente"].iloc[0]

print(f"Setor {setor_alvo}: Média antes = {media_antes}h | Média depois = {media_depois}h")
assert media_antes != media_depois, "A média não se alterou!"
print("ANOMALIA COMPROVADA: A média de tempo de atendimento mudou ao lançar peças, sem alteração de datas!")

## 2. Solução proposta: Segregação de granularidade

Para assegurar que **cada ordem de serviço encerrada tenha rigorosamente peso 1 na média** (mesmo que possua zero, uma ou dezenas de peças consumidas), as peças devem ser previamente consolidadas por ordem antes do agrupamento setorial:

```sql
WITH pecas_por_ordem AS (
    SELECT
        ordem_servico_id,
        SUM(quantidade) AS total_pecas_os
    FROM industria.itens_ordem
    GROUP BY ordem_servico_id
)
SELECT
    e.setor,
    COALESCE(SUM(p.total_pecas_os), 0) AS total_pecas_consumidas,
    ROUND(AVG(EXTRACT(EPOCH FROM (o.encerrada_em - o.aberta_em)) / 3600.0)::numeric, 4) AS tempo_medio_horas
FROM industria.ordens_servico o
JOIN industria.equipamentos e ON e.id = o.equipamento_id
LEFT JOIN pecas_por_ordem p ON p.ordem_servico_id = o.id
WHERE o.status = 'ENCERRADA'
GROUP BY e.setor
ORDER BY e.setor;
```

In [ ]:
# Validação da consulta corrigida
sql_painel_corrigido = """
WITH pecas_por_ordem AS (
    SELECT
        ordem_servico_id,
        SUM(quantidade) AS total_pecas_os
    FROM industria.itens_ordem
    GROUP BY ordem_servico_id
)
SELECT
    e.setor,
    COALESCE(SUM(p.total_pecas_os), 0) AS total_pecas_consumidas,
    ROUND(AVG(EXTRACT(EPOCH FROM (o.encerrada_em - o.aberta_em)) / 3600.0)::numeric, 4) AS tempo_medio_horas
FROM industria.ordens_servico o
JOIN industria.equipamentos e ON e.id = o.equipamento_id
LEFT JOIN pecas_por_ordem p ON p.ordem_servico_id = o.id
WHERE o.status = 'ENCERRADA'
GROUP BY e.setor
ORDER BY e.setor;
"""

with conectar_testes() as conn:
    df_corrigido_1 = pd.read_sql(sql_painel_corrigido, conn)

# Inserir mais um item na mesma OS
with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO industria.itens_ordem
                (ordem_servico_id, componente_id, quantidade, valor_unitario_aplicado)
            VALUES (%s, 3, 2, 50.00);
        """, (os_id,))
        conn.commit()

# Recalcular com a consulta corrigida
with conectar_testes() as conn:
    df_corrigido_2 = pd.read_sql(sql_painel_corrigido, conn)

print("Painel corrigido - Execução 1:")
display(df_corrigido_1)
print("Painel corrigido - Execução 2 (após novas peças):")
display(df_corrigido_2)

media_corr_1 = df_corrigido_1[df_corrigido_1["setor"] == setor_alvo]["tempo_medio_horas"].iloc[0]
media_corr_2 = df_corrigido_2[df_corrigido_2["setor"] == setor_alvo]["tempo_medio_horas"].iloc[0]

print(f"Média Corrigida 1: {media_corr_1}h | Média Corrigida 2: {media_corr_2}h")
assert media_corr_1 == media_corr_2, "A média corrigida variou indevidamente!"
print("SUCESSO ABSOLUTO: O total de peças aumentou e a média de tempo permaneceu rigorosamente estável!")

# Ficha técnica | INC-014

## 1. Hipótese e resultado esperado
O painel gerencial indicou alteração indevida no tempo médio de atendimento do setor após o lançamento de peças em ordens já encerradas.
- **Hipótese:** Agregação de entidades em granularidades diferentes (ordens e itens) no mesmo nível de GROUP BY, gerando duplicação de linhas e distorção na ponderação da função `AVG()`.
- **Resultado esperado:** Cada ordem de serviço encerrada deve ter peso exatamente 1 no cálculo da média setorial de tempo, independentemente de possuir zero ou múltiplas peças associadas.

## 2. Procedimento de investigação
1. Análise da consulta SQL fornecida pelo cliente para a geração do painel.
2. Identificação da duplicação de linhas no conjunto resultante do `LEFT JOIN` entre `ordens_servico` e `itens_ordem`.
3. Medição da média setorial de tempo antes e após a inserção experimental de peças em uma OS encerrada.
4. Elaboração de modelo analítico demonstrando a ponderação desbalanceada gerada pelas réplicas.

## 3. Evidências e resultados
- A execução da query do cliente após a inserção de peças alterou imediatamente o tempo médio de atendimento do setor, mesmo mantendo intactas as datas de abertura e encerramento.
- A consulta reestruturada com agregação prévia de peças em CTE manteve a média do setor 100% estável e idêntica, computando corretamente o incremento de peças consumidas.

## 4. Conclusão
Incidente plenamente diagnosticado e comprovado. A inconsistência decorre de falha metodológica clássica de agregação em consultas relacionais.

## 5. Solução proposta e validação
1. Substituir a consulta do painel pela versão com CTE pré-agregando `itens_ordem`.
2. Validar lançando peças de teste e certificando que o tempo médio do setor permaneça imutável.